# Jiaozi — Full Chain (Module 1 → 2 → 3 → 4)

Natural-language query **+** a HuggingFace dataset → a full model recommendation **+** a runnable training project. Built to run on **Google Colab**.

| Module | Does | Needs |
|---|---|---|
| **M1** | LLM extracts `task_type` / `priority` / `constraints` from the query | an LLM API key (Qwen or OpenAI) |
| **M2** | analyzes the dataset → `data_size`, `num_classes`, `class_imbalance`, resolution + colour mode | dataset download |
| **M3** | hybrid RAG retrieval → Top-3 backbone/head/loss/optimizer/checkpoint **+ recipe** (image_size / lr / epochs / **augmentation**) | — |
| **M4** | generates a runnable PyTorch project that **consumes the recipe**, then trains it | — (template mode; no key) |

Branch: `fullchain-demo` (recipe layer + Module 4 augmentation consumption + CE-over-focal for class-imbalance).

> Sections 1–5 run on CPU. **Section 6 (training) needs a GPU runtime** — Runtime → Change runtime type → T4.

## 1. Clone + install

In [ ]:
!git clone --branch fullchain-demo https://github.com/Isso-W/Jiaozi.git
%cd Jiaozi

In [ ]:
# torch / torchvision / pandas / numpy / pillow / scikit-learn ship with Colab.
!pip -q install chromadb datasets networkx "openai>=2.0.0" sentence-transformers transformers

## 2. LLM key (Module 1 only)

Module 1 defaults to **Qwen / DashScope**. Paste your key below, or switch to OpenAI. Module 4 stays template-only (no key).

In [ ]:
import os

# --- Qwen / DashScope (default) ---
os.environ["JIAOZI_LLM_PROVIDER"] = "qwen"
os.environ["JIAOZI_DASHSCOPE_API_KEY"] = ""   # <-- paste DashScope key

# --- OR OpenAI instead (uncomment) ---
# os.environ["JIAOZI_LLM_PROVIDER"] = "openai"
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Module 4 code-gen: template only (deterministic, no LLM). Set to 'openai'/'qwen' to LLM-generate model.py.
os.environ.setdefault("M4_LLM_PROVIDER", "none")

assert os.environ.get("JIAOZI_DASHSCOPE_API_KEY") or os.environ.get("OPENAI_API_KEY"), \
    "Set a DashScope or OpenAI key above (Module 1 needs it)."
print("Module 1 provider:", os.environ["JIAOZI_LLM_PROVIDER"])

## 3. Run the full chain (M1 → M4)

Pick a query + dataset. This runs everything and writes a runnable project to `generated_project/`, then a 1-step **smoke** to prove the generated code executes.

Dataset ideas: `uoft-cs/cifar10` (RGB), `ylecun/mnist` (grayscale → watch the recipe drop colour jitter), or any `org/name[:subset]` HF image-classification dataset.

In [ ]:
from pipeline import run_pipeline

QUERY   = "fine-grained bird species classification, accuracy matters"
DATASET = "uoft-cs/cifar10"    # org/name[:subset]

result = run_pipeline(
    QUERY, DATASET,
    fmt="nl",
    module4_output="generated_project",   # M4 writes the runnable project here
    module4_skip_smoke=False,             # 1-step smoke = proof the generated code runs
)

## 4. What the chain decided

In [ ]:
import json

m3 = result["module3_input"]
print("task_type :", m3["task_type"])
print("data_size :", m3["data_size"], " priority:", m3["priority"])
print("data_stats:", m3["data_stats"])          # resolution_tier + colour_mode, straight from Module 2
print("constraints:", m3["constraints"])

mc = result["task_lists"][0]["model_config"]
print("\nTop-1:", mc["backbone"], "| checkpoint:", mc.get("pretrained_hf_id") or "(scratch)")
print("loss :", mc["loss"], "| finetune:", mc.get("finetune_strategy"))
print("image_size:", mc.get("image_size"), "| lr:", mc.get("learning_rate"), "| epochs:", mc.get("recommended_epochs"))

# --- Augmentation choice (the recipe's decision, in plain terms) ---
aug = mc["recipe"]["augmentation"]
inv = aug["invariance"]
on = lambda b: "ON " if b else "off"
print("\nAugmentation choice")
print("  strength tier   :", aug["tier"])
print("  schedule        :", aug["schedule"])
print("  horizontal flip :", on(inv["hflip"]))
print("  vertical flip   :", on(inv["vflip"]))
print("  90-deg rotation :", on(inv["rot90"]))
print("  colour jitter   :", on(inv["color"]))
print("  crop scale floor:", inv["crop_scale_min"])
print("  why:", mc["recipe_provenance"]["augmentation"])

## 5. The recipe reaches the generated code

Build the *actual* training transform from the generated project and read off its ops. The invariance mask is honoured op-by-op — e.g. grayscale data drops `ColorJitter`, fine-grained enforces a crop-scale floor.

In [ ]:
import sys, importlib, json

sys.path.insert(0, "generated_project")
for _m in ("train", "utils", "model", "smoke_data"):
    sys.modules.pop(_m, None)
import utils, train

cfg = utils.normalize_config(json.load(open("generated_project/configs.json"))[0])
t = train._build_image_transform(cfg, "train")
print("Train transform ops:", [type(o).__name__ for o in t.transforms])
print("Augmentation schedule:", train._augmentation_schedule(cfg))

## 6. Train on Colab (GPU)

Set the runtime to **GPU** first (Runtime → Change runtime type → T4). This regenerates the project with real training (`offline_smoke=false`) and fine-tunes the Top-1 config on the dataset — using the recipe's `image_size` / `lr` / `epochs` **and** the exact augmentation pipeline shown above. The live training log (per-epoch loss + val metric) prints below.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (set runtime to GPU)")

EPOCHS = 3   # bump for a longer run; set to None to use the recipe's recommended_epochs

# Regenerate the project with real-training code (offline_smoke=false).
rt = run_pipeline(
    QUERY, DATASET, fmt="nl",
    module4_output="generated_rt",
    module4_real_training=True,   # offline_smoke=false; skips the local smoke
)

# Fine-tune the Top-1 config on the real dataset — live training log below.
epochs_flag = f"--epochs {EPOCHS}" if EPOCHS else ""
!cd generated_rt && python run.py {epochs_flag}

---
**What just happened.** M1 read your query → M2 measured the dataset → M3 picked a backbone/loss/checkpoint and a deterministic recipe → M4 emitted a project whose training transform is built from that recipe. Swap `QUERY`/`DATASET` and re-run — the recommendation, recipe, and generated augmentation all change with the inputs.